In [0]:
# ---- SK-04 setup: build bronze.orders_dirty (bronze.orders itself is NOT modified) ----
from pyspark.sql import functions as F

SRC = "shopkart.bronze.orders"
DST = "shopkart.bronze.orders_dirty"
TS_COLS = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
           "order_delivered_customer_date", "order_estimated_delivery_date"]

# Every order gets a fixed "bucket" 0-999 from a hash of its id, so the same
# orders get the same problems on every run (reproducible before/after counts).
base = (spark.table(SRC)
        .drop("_ingested_at", "_source_file")
        .withColumn("_bucket", F.abs(F.xxhash64("order_id")) % 1000)
        .withColumn("_extracted_at", F.lit("2018-09-01 06:00:00")))   # when the source export ran

b = F.col("_bucket")
dirty = (base
    # Problem 1 + 2: NULL status (~1.5%) and messy case / spaces (~5%)
    .withColumn("order_status",
        F.when(b.between(20, 34), F.lit(None).cast("string"))
         .when(b.between(35, 49), F.upper("order_status"))                               # "DELIVERED"
         .when(b.between(50, 64), F.concat(F.initcap("order_status"), F.lit(" ")))       # "Delivered "
         .when(b.between(65, 84), F.concat(F.lit("  "), F.initcap("order_status")))      # "  Shipped"
         .otherwise(F.col("order_status")))
    # Problem 3: delivered 3 days BEFORE purchase (~1% of orders that have a delivery date)
    .withColumn("order_delivered_customer_date",
        F.when(b.between(85, 94) & F.col("order_delivered_customer_date").isNotNull(),
               F.col("order_purchase_timestamp") - F.expr("INTERVAL 3 DAYS"))
         .otherwise(F.col("order_delivered_customer_date"))))

# Problem 4a: exact duplicate rows (~1%)
exact_dups = dirty.filter(b < 10)

# Problem 4b: key duplicates (~1%): an OLDER snapshot of the same order_id,
# still "processing" and not yet delivered. Your dedup must keep the newest one.
stale_dups = (dirty.filter(b.between(10, 19))
    .withColumn("order_status", F.lit("processing"))
    .withColumn("order_delivered_carrier_date", F.lit(None).cast("timestamp"))
    .withColumn("order_delivered_customer_date", F.lit(None).cast("timestamp"))
    .withColumn("_extracted_at", F.lit("2018-08-25 06:00:00")))

result = (dirty.unionByName(exact_dups).unionByName(stale_dups)
    # Problem 5: dates arrive as strings, like a raw export
    .select(*[F.date_format(c, "yyyy-MM-dd HH:mm:ss").alias(c) if c in TS_COLS else F.col(c)
              for c in dirty.columns])
    .drop("_bucket"))

result.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(DST)
print(f"Wrote {DST}: {spark.table(DST).count():,} rows")

In [0]:
# ---- SK-04: "before" profile of bronze.orders_dirty (run this again on silver.orders for "after") ----
from pyspark.sql import functions as F

df = spark.table("shopkart.bronze.orders_dirty")
total = df.count()
profile = {
    "total rows":                total,
    "distinct order_id":         df.select("order_id").distinct().count(),
    "exact duplicate rows":      total - df.distinct().count(),
    "NULL order_status":         df.filter(F.col("order_status").isNull()).count(),
    "non-normalised status":     df.filter(F.col("order_status") != F.lower(F.trim("order_status"))).count(),
    "delivered before purchase": df.filter(F.to_timestamp("order_delivered_customer_date")
                                           < F.to_timestamp("order_purchase_timestamp")).count(),
}
profile["key duplicate rows (total - distinct ids)"] = total - profile["distinct order_id"]
for k, v in profile.items():
    print(f"{k:<45} {v:>7,}")